In [1]:
import h5py
import numpy as np
import pandas as pd
from scipy.sparse import csc_matrix, issparse
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import os

# --- 0. Setup & Helper Function ---

# Ensure a 'figures' directory exists to save plots
if not os.path.exists("figures"):
    os.makedirs("figures")
    print("Created 'figures' directory.")

# Gene list for Plasticity Score
genes_of_interest = [
    'Fos', 'Fosb', 'Junb', 'Egr1', 'Egr2', 'Npas4', 'Arc', 
    'Nr4a1', 'Nr4a2', 'Bdnf', 'Nptx2', 'Syt4', 'Camk2g', 
    'Rgs4', 'Dusp1', 'Atf3', 'Crem', 'Gadd45b', 'Gadd45g', 'Prkcz'
]

# Helper function from your notebook to load the HDF5 matrix
def load_allen_smartseq_h5(path, group="data/t_exon"):
    print(f"Loading matrix from: {path}")
    with h5py.File(path, "r") as f:
        genes = [g.decode() for g in f["gene_names"][:]]
        # **FIXED**: Iterating over 's' in f["sample_names"]
        cells = [s.decode() for s in f["sample_names"][:]] 
        
        G, C = len(genes), len(cells)
        print(f"Found {C} cells and {G} genes in HDF5.")

        grp = f[group]
        dims = tuple(grp["dims"][:])
        i = grp["i"][:]
        p = grp["p"][:]
        x = grp["x"][:]

        X = csc_matrix((x, i, p), shape=dims)

        ncols = len(p) - 1
        if ncols == G:
            print("Matrix orientation: Cells x Genes")
            pass
        elif ncols == C:
            print("Matrix orientation: Genes x Cells. Transposing...")
            X = X.T
        else:
            print("Warning: Could not infer orientation from p-length. Checking dims...")
            if dims[0] == C and dims[1] == G:
                print("Matrix orientation: Cells x Genes")
                pass
            elif dims[0] == G and dims[1] == C:
                print("Matrix orientation: Genes x Cells. Transposing...")
                X = X.T
            else:
                raise ValueError(
                    f"Can't infer orientation: dims={dims}, genes={G}, cells={C}, ncols={ncols}"
                )

    return ad.AnnData(
        X=X,
        obs=pd.DataFrame(index=cells),
        var=pd.DataFrame(index=genes)
    )

print("--- Analysis Started ---")

# --- 1. Load & Prepare Data ---
print("[1/5] Loading data...")
try:
    adata = load_allen_smartseq_h5("Data/expression_matrix.hdf5")
    meta = pd.read_csv("Data/metadata.csv", low_memory=False)
except FileNotFoundError as e:
    print(f"Error: Could not find data files. Make sure 'Data/expression_matrix.hdf5' and 'Data/metadata.csv' exist.")
    exit()

# Clean keys and index by sample_name
meta["sample_name"] = meta["sample_name"].astype(str).str.strip()
meta = meta.set_index("sample_name")

# Align metadata to adata
in_meta = adata.obs_names.isin(meta.index)
print(f"      Matched {in_meta.sum():,} / {adata.n_obs:,} cells to metadata.")
meta_aligned = meta.reindex(adata.obs_names)
adata.obs = adata.obs.join(meta_aligned)

# --- 2. Filter Glutamatergic Cells ---
print("[2/5] Filtering for Glutamatergic neurons...")
mask_glut = adata.obs["class_label"].astype(str) == 'Glutamatergic'
adata_glut = adata[mask_glut].copy()

if adata_glut.n_obs == 0:
    print("CRITICAL ERROR: Filtered to 0 Glutamatergic neurons. Check metadata.")
    exit()
else:
    print(f"      Filtered to {adata_glut.n_obs:,} Glutamatergic neurons.")

# --- 3. Calculate Plasticity Score ---
print("[3/5] Normalizing data and calculating Plasticity Score...")
# CPM normalization and log transform
sc.pp.normalize_total(adata_glut, target_sum=1e6)
sc.pp.log1p(adata_glut)

# Check which genes are present
found_genes = [g for g in genes_of_interest if g in adata_glut.var_names]

if len(found_genes) < len(genes_of_interest):
    missing = set(genes_of_interest) - set(found_genes)
    print(f"      Warning: Missing {len(missing)} genes: {missing}")
    
print(f"      Using {len(found_genes)}/{len(genes_of_interest)} plasticity genes to calculate score.")

if not found_genes:
    print("CRITICAL ERROR: No plasticity genes found in the dataset. Cannot calculate score.")
else:
    # Calculate Z-score-based plasticity score
    X = adata_glut[:, found_genes].X
    X = X.toarray() if issparse(X) else np.asarray(X)
    
    # Calculate mean and std per gene
    gene_mean = X.mean(axis=0)
    gene_std = X.std(axis=0)
    
    # Z-score each gene across cells
    # The minimum required 'flavor' for sc.pp.highly_variable_genes is the one that avoids the error you encountered. 
    # The default flavor in older versions that caused the error is often 'seurat_v3' or 'seurat'. We will switch to 'cell_ranger' or 'seurat' (which you had in the error message)
    # The error message said "ValueError: `flavor` needs to be "seurat" or "cell_ranger"". This means the version of scanpy you are running *requires* the flavor argument.
    expr_z = (X - gene_mean) / (gene_std + 1e-8)
    
    # Per-cell plasticity score = mean z across genes
    adata_glut.obs["plasticity_score"] = expr_z.mean(axis=1)
    print(f"      Plasticity score calculated. Range: {adata_glut.obs['plasticity_score'].min():.2f} to {adata_glut.obs['plasticity_score'].max():.2f}")

# --- 4. Generate UMAP Embedding ---
print("[4/5] Running PCA and UMAP...")
# **FIXED**: Explicitly setting flavor to 'seurat' to satisfy scanpy version requirement from the traceback.
sc.pp.highly_variable_genes(adata_glut, n_top_genes=2000, flavor='seurat')
sc.pp.pca(adata_glut)
sc.pp.neighbors(adata_glut)
sc.tl.umap(adata_glut)
print("      UMAP calculation complete.")

# --- 5. Generate & Save Subfigures ---
print("[5/5] Generating and saving figures...")

# Set up aesthetics
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10

# --- Figure 1A: Violin Plot ---
plt.figure(figsize=(15, 7))
subclass_order = np.sort(adata_glut.obs['subclass_label'].unique().astype(str))
sns.violinplot(
    data=adata_glut.obs, 
    x="subclass_label", 
    y="plasticity_score", 
    cut=0, 
    inner='quartile', 
    scale='width',
    order=subclass_order
)
plt.title("Figure 1A: Plasticity Score Distribution by Subclass (Glutamatergic)", loc='left')
plt.ylabel("Plasticity Score")
plt.xlabel("Glutamatergic Subclass")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("figures/figure_1A_violin_glutamatergic.png")
plt.close()
print("      Saved Figure 1A (violin plot).")

# --- Figure 1B: Bar Plot ---
avg_score = (
    adata_glut.obs.groupby("subclass_label")["plasticity_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 10))
sns.barplot(
    x=avg_score.values, 
    y=avg_score.index, 
    orient='h',
    palette='magma'
)
plt.title("Figure 1B: Mean Plasticity Score by Subclass (Glutamatergic)", loc='left')
plt.xlabel("Mean Plasticity Score")
plt.ylabel("Glutamatergic Subclass")
plt.tight_layout()
plt.savefig("figures/figure_1B_barplot_glutamatergic.png")
plt.close()
print("      Saved Figure 1B (bar plot).")

# --- Figure 1C: UMAP by Subclass ---
unique_subclasses = adata_glut.obs['subclass_label'].unique().astype(str)
n_subclasses = len(unique_subclasses)
palette = sc.pl.palettes.default_102
legend_fontsize = 5 if n_subclasses > 30 else 8

# Using sc.pl.umap with the save argument to handle figure generation
sc.pl.umap(
    adata_glut, 
    color="subclass_label", 
    title="Figure 1C: UMAP by Subclass (Glutamatergic)", 
    palette=palette,
    legend_loc='on data',
    legend_fontsize=legend_fontsize,
    frameon=False,
    show=False,
    save="_1C_umap_subclass.png"
)
# Ensure the file path is correct after scanpy save
if os.path.exists("./umap_1C_umap_subclass.png"):
    os.rename("./umap_1C_umap_subclass.png", "figures/figure_1C_umap_subclass.png")
elif os.path.exists("figures/umap_1C_umap_subclass.png"):
     os.rename("figures/umap_1C_umap_subclass.png", "figures/figure_1C_umap_subclass.png")

print("      Saved Figure 1C (UMAP by subclass).")


# --- Figure 1D: UMAP by Plasticity Score ---
sc.pl.umap(
    adata_glut, 
    color="plasticity_score", 
    title="Figure 1D: UMAP by Plasticity Score (Glutamatergic)",
    cmap="magma",
    frameon=False,
    show=False,
    save="_1D_umap_plasticity.png"
)
# Ensure the file path is correct after scanpy save
if os.path.exists("./umap_1D_umap_plasticity.png"):
    os.rename("./umap_1D_umap_plasticity.png", "figures/figure_1D_umap_plasticity.png")
elif os.path.exists("figures/umap_1D_umap_plasticity.png"):
    os.rename("figures/umap_1D_umap_plasticity.png", "figures/figure_1D_umap_plasticity.png")
    
print("      Saved Figure 1D (UMAP by plasticity score).")

# --- 6. Save Processed Data ---
print("Saving processed AnnData object to 'adata_glut_processed.h5ad'...")
adata_glut.write('adata_glut_processed.h5ad')
print("      Saved processed data.")

print("--- Analysis Complete. Figures saved to 'figures/' directory. ---")

--- Analysis Started ---
[1/5] Loading data...
Loading matrix from: Data/expression_matrix.hdf5
Found 73363 cells and 45768 genes in HDF5.
Matrix orientation: Genes x Cells. Transposing...
      Matched 73,347 / 73,363 cells to metadata.
[2/5] Filtering for Glutamatergic neurons...
      Filtered to 48,796 Glutamatergic neurons.
[3/5] Normalizing data and calculating Plasticity Score...


/home/peter/.local/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


      Using 20/20 plasticity genes to calculate score.
      Plasticity score calculated. Range: -1.84 to 3.01
[4/5] Running PCA and UMAP...
      UMAP calculation complete.
[5/5] Generating and saving figures...


/tmp/ipykernel_11013/3609324902.py:158: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(


      Saved Figure 1A (violin plot).


/tmp/ipykernel_11013/3609324902.py:184: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


      Saved Figure 1B (bar plot).
      Saved Figure 1C (UMAP by subclass).
      Saved Figure 1D (UMAP by plasticity score).
Saving processed AnnData object to 'adata_glut_processed.h5ad'...
      Saved processed data.
--- Analysis Complete. Figures saved to 'figures/' directory. ---


In [3]:
import h5py
import numpy as np
import pandas as pd
from scipy.sparse import csc_matrix, issparse
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import os

# ============================================================
# 0. Setup & Helper Function
# ============================================================

FIG_DIR = "figures"

if not os.path.exists(FIG_DIR):
    os.makedirs(FIG_DIR)
    print(f"Created '{FIG_DIR}' directory.")

# Global plotting style
sns.set_theme(style="white")  # no grid
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "pdf.fonttype": 42,  # editable text in Illustrator
    "ps.fonttype": 42,
})

# Gene list for Plasticity Score
genes_of_interest = [
    'Fos', 'Fosb', 'Junb', 'Egr1', 'Egr2', 'Npas4', 'Arc',
    'Nr4a1', 'Nr4a2', 'Bdnf', 'Nptx2', 'Syt4', 'Camk2g',
    'Rgs4', 'Dusp1', 'Atf3', 'Crem', 'Gadd45b', 'Gadd45g', 'Prkcz'
]


def load_allen_smartseq_h5(path, group="data/t_exon"):
    """Load Allen SmartSeq HDF5 into an AnnData object."""
    print(f"Loading matrix from: {path}")
    with h5py.File(path, "r") as f:
        genes = [g.decode() for g in f["gene_names"][:]]
        cells = [s.decode() for s in f["sample_names"][:]]

        G, C = len(genes), len(cells)
        print(f"Found {C:,} cells and {G:,} genes in HDF5.")

        grp = f[group]
        dims = tuple(grp["dims"][:])
        i = grp["i"][:]
        p = grp["p"][:]
        x = grp["x"][:]

        X = csc_matrix((x, i, p), shape=dims)

        ncols = len(p) - 1
        if ncols == G:
            print("Matrix orientation: Cells x Genes")
        elif ncols == C:
            print("Matrix orientation: Genes x Cells. Transposing...")
            X = X.T
        else:
            print("Warning: Could not infer orientation from p-length. Checking dims...")
            if dims[0] == C and dims[1] == G:
                print("Matrix orientation: Cells x Genes")
            elif dims[0] == G and dims[1] == C:
                print("Matrix orientation: Genes x Cells. Transposing...")
                X = X.T
            else:
                raise ValueError(
                    f"Can't infer orientation: dims={dims}, genes={G}, "
                    f"cells={C}, ncols={ncols}"
                )

    return ad.AnnData(
        X=X,
        obs=pd.DataFrame(index=cells),
        var=pd.DataFrame(index=genes)
    )


print("--- Analysis Started ---")

# ============================================================
# 1. Load & Prepare Data
# ============================================================
print("[1/6] Loading data...")
try:
    adata = load_allen_smartseq_h5("Data/expression_matrix.hdf5")
    meta = pd.read_csv("Data/metadata.csv", low_memory=False)
except FileNotFoundError as e:
    print("Error: Could not find data files. "
          "Make sure 'Data/expression_matrix.hdf5' and 'Data/metadata.csv' exist.")
    raise

# Clean keys and index by sample_name
meta["sample_name"] = meta["sample_name"].astype(str).str.strip()
meta = meta.set_index("sample_name")

# Align metadata to adata
in_meta = adata.obs_names.isin(meta.index)
print(f"      Matched {in_meta.sum():,} / {adata.n_obs:,} cells to metadata.")
meta_aligned = meta.reindex(adata.obs_names)
adata.obs = adata.obs.join(meta_aligned)

# ============================================================
# 2. Filter Glutamatergic Cells
# ============================================================
print("[2/6] Filtering for Glutamatergic neurons...")
mask_glut = adata.obs["class_label"].astype(str) == "Glutamatergic"
adata_glut = adata[mask_glut].copy()

if adata_glut.n_obs == 0:
    raise RuntimeError("Filtered to 0 Glutamatergic neurons. Check metadata.")
else:
    print(f"      Filtered to {adata_glut.n_obs:,} Glutamatergic neurons.")

# ============================================================
# 3. Calculate Plasticity Score
# ============================================================
print("[3/6] Normalizing data and calculating Plasticity Score...")

# CPM normalization and log transform
sc.pp.normalize_total(adata_glut, target_sum=1e6)
sc.pp.log1p(adata_glut)

# Check which genes are present
found_genes = [g for g in genes_of_interest if g in adata_glut.var_names]
if len(found_genes) < len(genes_of_interest):
    missing = sorted(set(genes_of_interest) - set(found_genes))
    print(f"      Warning: Missing {len(missing)} genes: {missing}")

print(f"      Using {len(found_genes)}/{len(genes_of_interest)} plasticity genes to calculate score.")

if not found_genes:
    raise RuntimeError("No plasticity genes found in the dataset. Cannot calculate score.")

# Extract expression for plasticity genes
X = adata_glut[:, found_genes].X
X = X.toarray() if issparse(X) else np.asarray(X)

# Z-score per gene across cells
gene_mean = X.mean(axis=0)
gene_std = X.std(axis=0)
expr_z = (X - gene_mean) / (gene_std + 1e-8)

# Per-cell plasticity score = mean z across genes
adata_glut.obs["plasticity_score"] = expr_z.mean(axis=1)
print(f"      Plasticity score calculated. Range: "
      f"{adata_glut.obs['plasticity_score'].min():.2f} to "
      f"{adata_glut.obs['plasticity_score'].max():.2f}")

# ============================================================
# 4. Dimensionality Reduction (PCA + UMAP)
# ============================================================
print("[4/6] Running PCA and UMAP...")
sc.pp.highly_variable_genes(adata_glut, n_top_genes=2000, flavor="seurat")
sc.pp.pca(adata_glut)
sc.pp.neighbors(adata_glut)
sc.tl.umap(adata_glut)
print("      UMAP calculation complete.")

# Build a small DataFrame for plotting UMAPs
umap_df = pd.DataFrame(
    adata_glut.obsm["X_umap"],
    columns=["UMAP1", "UMAP2"],
    index=adata_glut.obs_names,
)
umap_df["subclass_label"] = adata_glut.obs["subclass_label"].astype(str).values
umap_df["plasticity_score"] = adata_glut.obs["plasticity_score"].values

# ============================================================
# 5. Figure 1A & 1B: Plasticity distributions by subclass
# ============================================================
print("[5/6] Generating Figure 1A and 1B...")

# Compute mean plasticity per subclass first – use same order for A & B
avg_score = (
    adata_glut.obs.groupby("subclass_label")["plasticity_score"]
    .mean()
    .sort_values(ascending=False)
)
subclass_order = avg_score.index.astype(str).tolist()

# --- Figure 1A: Violin Plot ---
plt.figure(figsize=(14, 6))
ax = sns.violinplot(
    data=adata_glut.obs,
    x="subclass_label",
    y="plasticity_score",
    cut=0,
    inner="quartile",
    scale="width",
    order=subclass_order,
    linewidth=0.7,
    color="#4C72B0",
)
ax.set_title("Figure 1A: Plasticity score distribution by subclass (Glutamatergic)",
             loc="left")
ax.set_xlabel("Glutamatergic subclass")
ax.set_ylabel("Plasticity score")
for label in ax.get_xticklabels():
    label.set_rotation(90)
    label.set_ha("center")
sns.despine()
plt.tight_layout()

for ext in ("png", "svg"):
    plt.savefig(
        os.path.join(FIG_DIR, f"figure_1A_violin_glutamatergic.{ext}"),
        bbox_inches="tight",
    )
plt.close()
print("      Saved Figure 1A (violin plot, PNG & SVG).")

# --- Figure 1B: Bar Plot (same order, colored by score) ---
plt.figure(figsize=(6, 8))
# Color bars by score (magma, centered near 0)
cmap = sns.color_palette("magma", as_cmap=True)
normed = (avg_score - avg_score.min()) / (avg_score.max() - avg_score.min() + 1e-8)
colors = cmap(normed.values)

ax = plt.gca()
ax.barh(
    y=avg_score.index,
    width=avg_score.values,
    color=colors,
    edgecolor="none",
)
ax.set_title("Figure 1B: Mean plasticity score by subclass (Glutamatergic)",
             loc="left")
ax.set_xlabel("Mean plasticity score")
ax.set_ylabel("Glutamatergic subclass")
ax.invert_yaxis()  # highest at top
sns.despine()
plt.tight_layout()

for ext in ("png", "svg"):
    plt.savefig(
        os.path.join(FIG_DIR, f"figure_1B_barplot_glutamatergic.{ext}"),
        bbox_inches="tight",
    )
plt.close()
print("      Saved Figure 1B (bar plot, PNG & SVG).")

## ============================================================
# 6. Figure 1C & 1D: UMAPs (clean aspect ratio + nicer colors)
# ============================================================
print("[6/6] Generating Figure 1C and 1D (UMAPs)...")

# --- Figure 1C: UMAP by subclass ---
plt.figure(figsize=(6.5, 6))
unique_subclasses = subclass_order  # same order as above
n_subclasses = len(unique_subclasses)

# Use a nicer discrete palette for subclasses
palette = sns.color_palette("tab20", n_colors=n_subclasses)
color_map = dict(zip(unique_subclasses, palette))

for subclass in unique_subclasses:
    subset = umap_df[umap_df["subclass_label"] == subclass]
    plt.scatter(
        subset["UMAP1"],
        subset["UMAP2"],
        s=3,
        color=color_map[subclass],
        label=subclass,
        linewidths=0,
        alpha=0.9,
    )

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Figure 1C: UMAP colored by subclass (Glutamatergic)", loc="left")

# Legend INSIDE the axes so aspect ratio is controlled only by figsize
ncol = 2 if n_subclasses <= 24 else 3
leg = plt.legend(
    title="Subclass",
    loc="upper right",
    frameon=True,
    framealpha=0.8,
    markerscale=3,
    ncol=ncol,
    borderpad=0.3,
    handletextpad=0.3,
)
leg.get_frame().set_linewidth(0.5)

sns.despine()
plt.tight_layout()

for ext in ("png", "svg"):
    plt.savefig(
        os.path.join(FIG_DIR, f"figure_1C_umap_subclass.{ext}"),
        # no bbox_inches="tight" → same width/height as 1D
    )
plt.close()
print("      Saved Figure 1C (UMAP by subclass, PNG & SVG).")

# --- Figure 1D: UMAP by plasticity score ---
plt.figure(figsize=(6.5, 6))
scat = plt.scatter(
    umap_df["UMAP1"],
    umap_df["UMAP2"],
    c=umap_df["plasticity_score"],
    s=3,
    cmap="magma",
    linewidths=0,
    alpha=0.9,
)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Figure 1D: UMAP colored by plasticity score (Glutamatergic)", loc="left")

cbar = plt.colorbar(scat, pad=0.02)
cbar.set_label("Plasticity score")

sns.despine()
plt.tight_layout()

for ext in ("png", "svg"):
    plt.savefig(
        os.path.join(FIG_DIR, f"figure_1D_umap_plasticity.{ext}"),
        # also no bbox_inches="tight"
    )
plt.close()
print("      Saved Figure 1D (UMAP by plasticity score, PNG & SVG).")

# ============================================================
# 7. Save Processed Data
# ============================================================
print("Saving processed AnnData object to 'adata_glut_processed.h5ad'...")
adata_glut.write("adata_glut_processed.h5ad")
print("      Saved processed data.")

print("--- Analysis Complete. Figures saved to 'figures/' ---")


--- Analysis Started ---
[1/6] Loading data...
Loading matrix from: Data/expression_matrix.hdf5
Found 73,363 cells and 45,768 genes in HDF5.
Matrix orientation: Genes x Cells. Transposing...
      Matched 73,347 / 73,363 cells to metadata.
[2/6] Filtering for Glutamatergic neurons...
      Filtered to 48,796 Glutamatergic neurons.
[3/6] Normalizing data and calculating Plasticity Score...
      Using 20/20 plasticity genes to calculate score.
      Plasticity score calculated. Range: -1.84 to 3.01
[4/6] Running PCA and UMAP...
      UMAP calculation complete.
[5/6] Generating Figure 1A and 1B...


/tmp/ipykernel_11013/2883837313.py:196: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


      Saved Figure 1A (violin plot, PNG & SVG).
      Saved Figure 1B (bar plot, PNG & SVG).
[6/6] Generating Figure 1C and 1D (UMAPs)...
      Saved Figure 1C (UMAP by subclass, PNG & SVG).
      Saved Figure 1D (UMAP by plasticity score, PNG & SVG).
Saving processed AnnData object to 'adata_glut_processed.h5ad'...
      Saved processed data.
--- Analysis Complete. Figures saved to 'figures/' ---
